# Customer Segmentation — RFM Analysis + K-Means Clustering
**Dataset:** Online Retail Transactions (UK-based gift retailer, 2009–2011)  
**Kaggle Reference:** [Online Retail II Data Set from UCI ML Repository](https://www.kaggle.com/datasets/mashlyn/online-retail-ii-uci)  
**Objective:** Segment customers using Recency, Frequency & Monetary (RFM) scores, then apply K-Means clustering to discover actionable customer groups.

---
### Project Modules
| # | Module | Description |
|---|--------|-------------|
| 1 | Data Loading & Exploration | Load dataset, understand structure |
| 2 | Data Cleaning & Preprocessing | Handle missing values, fix data types |
| 3 | RFM Feature Engineering | Calculate R, F, M for each customer |
| 4 | RFM Scoring & Segmentation | Score and label customer tiers |
| 5 | K-Means Clustering | Machine learning based grouping |
| 6 | Cluster Profiling & Business Insights | Understand each segment |
| 7 | Visualization Dashboard (Python) | Charts for all KPIs |
| 8 | Excel Report Generator | Professional multi-sheet Excel report |

## MODULE 1 — Data Loading & Initial Exploration

In [ ]:
# standard imports — keeping it clean and readable
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# plot style
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans'
})
PALETTE = ['#2C3E50','#2980B9','#27AE60','#E74C3C','#F39C12','#8E44AD','#16A085','#D35400']

print("All libraries loaded successfully!")

All libraries loaded successfully!


In [ ]:
# load the dataset — this is the Online Retail style transactional data
df_raw = pd.read_csv('/content/online_retail_rfm (1).csv', parse_dates=['InvoiceDate'])

print(f"Shape          : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Date range     : {df_raw['InvoiceDate'].min().date()} → {df_raw['InvoiceDate'].max().date()}")
print(f"Unique customers: {df_raw['CustomerID'].nunique():,}")
print(f"Unique invoices : {df_raw['InvoiceNo'].nunique():,}")
print(f"Countries       : {df_raw['Country'].nunique()}")
print()
print("─" * 55)
print("Column overview:")
print(df_raw.dtypes)

Shape          : 90,097 rows × 8 columns
Date range     : 2009-12-01 → 2011-12-09
Unique customers: 3,977
Unique invoices : 20,000
Countries       : 8

───────────────────────────────────────────────────────
Column overview:
InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
UnitPrice             float64
CustomerID             object
Country                object
dtype: object


In [ ]:
# quick peek at first few records
df_raw.head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,INV536365,84946,HAND WARMER UNION JACK,19,2010-03-25 08:47:00,1.85,C12619,United Kingdom
1,INV536365,22356,JUMBO STORAGE BAG SUKI,14,2010-03-25 08:47:00,2.08,C12619,United Kingdom
2,INV536365,85099C,RED WOOLLY HOTTIE WHITE HEART,2,2010-03-25 08:47:00,3.39,C12619,United Kingdom
3,INV536365,85099B,KNITTED UNION FLAG HOT WATER BOTTLE,1,2010-03-25 08:47:00,3.39,C12619,United Kingdom
4,INV536365,84406B,CREAM CUPID HEARTS COAT HANGER,3,2010-03-25 08:47:00,2.75,C12619,United Kingdom
5,INV536366,22137,CERAMIC STORAGE JAR POLKADOT,7,2010-07-27 16:38:00,4.95,C10895,United Kingdom
6,INV536367,84946,HAND WARMER UNION JACK,11,2011-09-27 19:34:00,1.85,C12932,United Kingdom
7,INV536367,22355,ROUND SNACK BOXES SET OF4 WOODLAND,9,2011-09-27 19:34:00,4.25,C12932,United Kingdom
8,INV536367,84970S,HAND WARMER RED POLKA DOT,5,2011-09-27 19:34:00,1.85,C12932,United Kingdom
9,INV536367,85123A,WHITE HANGING HEART T-LIGHT HOLDER,7,2011-09-27 19:34:00,2.55,C12932,United Kingdom


In [ ]:
# high-level statistics
df_raw.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
InvoiceNo,90097,20000,INV543931,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,90097,18,22197,9111,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,90097,20,NATURAL SLATE HEART CHALKBOARD,4577,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,90097.0,NaN,NaN,NaN,15.519351,1.0,8.0,16.0,23.0,30.0,8.679161
InvoiceDate,90097,NaN,NaN,NaN,2010-12-07 03:18:47.155843072,2009-12-01 09:14:00,2010-06-04 14:50:00,2010-12-08 15:01:00,2011-06-10 13:02:00,2011-12-09 20:05:00,NaN
UnitPrice,90097.0,NaN,NaN,NaN,2.8288,0.55,1.25,2.55,4.25,7.65,1.80144
CustomerID,90097,3977,C10751,82,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,90097,8,United Kingdom,53323,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## MODULE 2 — Data Cleaning & Preprocessing

In [ ]:
# check for nulls and obvious issues before cleaning
print("Missing values per column:")
print(df_raw.isnull().sum())
print()
print(f"Rows with missing CustomerID : {df_raw['CustomerID'].isnull().sum():,}")
print(f"Rows with Quantity <= 0      : {(df_raw['Quantity'] <= 0).sum():,}")
print(f"Rows with UnitPrice <= 0     : {(df_raw['UnitPrice'] <= 0).sum():,}")

Missing values per column:
InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64

Rows with missing CustomerID : 0
Rows with Quantity <= 0      : 0
Rows with UnitPrice <= 0     : 0


In [ ]:
# cleaning steps — removing bad records
df = df_raw.copy()

# drop rows with no customer id — we can't attribute them to anyone
df.dropna(subset=['CustomerID'], inplace=True)

# remove cancelled orders (negative quantities) and free items
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# calculate total spend per line item
df['TotalPrice'] = df['Quantity'] * df['UnitPrice']

# make sure CustomerID is treated as a string label
df['CustomerID'] = df['CustomerID'].astype(str)

print(f"Rows after cleaning : {len(df):,}")
print(f"Customers retained  : {df['CustomerID'].nunique():,}")
print(f"Total revenue       : ${df['TotalPrice'].sum():,.2f}")
print()
print("Sample cleaned data:")
df[['InvoiceNo','CustomerID','Description','Quantity','UnitPrice','TotalPrice','InvoiceDate']].head(5)

Rows after cleaning : 90,097
Customers retained  : 3,977
Total revenue       : $3,955,270.36

Sample cleaned data:


,InvoiceNo,CustomerID,Description,Quantity,UnitPrice,TotalPrice,InvoiceDate
0,INV536365,C12619,HAND WARMER UNION JACK,19,1.85,35.15,2010-03-25 08:47:00
1,INV536365,C12619,JUMBO STORAGE BAG SUKI,14,2.08,29.12,2010-03-25 08:47:00
2,INV536365,C12619,RED WOOLLY HOTTIE WHITE HEART,2,3.39,6.78,2010-03-25 08:47:00
3,INV536365,C12619,KNITTED UNION FLAG HOT WATER BOTTLE,1,3.39,3.39,2010-03-25 08:47:00
4,INV536365,C12619,CREAM CUPID HEARTS COAT HANGER,3,2.75,8.25,2010-03-25 08:47:00


In [ ]:
# country distribution — good to understand geographic spread
country_dist = df.groupby('Country')['TotalPrice'].sum().sort_values(ascending=False).reset_index()
country_dist.columns = ['Country','Revenue']
country_dist['Revenue_pct'] = (country_dist['Revenue'] / country_dist['Revenue'].sum() * 100).round(2)
print("Revenue by Country:")
print(country_dist.head(10).to_string(index=False))

Revenue by Country:
       Country    Revenue  Revenue_pct
United Kingdom 2336940.03        59.08
       Germany  423974.79        10.72
        France  346172.88         8.75
         Spain  205452.59         5.19
   Switzerland  165415.76         4.18
     Australia  165140.38         4.18
   Netherlands  156812.01         3.96
       Belgium  155361.92         3.93


## MODULE 3 — RFM Feature Engineering

In [ ]:
# snapshot date — one day after the last transaction in the dataset
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot date (reference point): {snapshot_date.date()}")

# build the RFM table — one row per customer
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

# round monetary to 2 decimal places
rfm['Monetary'] = rfm['Monetary'].round(2)

print(f"\nRFM table shape: {rfm.shape}")
print()
print("RFM summary statistics:")
print(rfm[['Recency','Frequency','Monetary']].describe().round(2).to_string())

Snapshot date (reference point): 2011-12-10

RFM table shape: (3977, 4)

RFM summary statistics:
       Recency  Frequency  Monetary
count  3977.00    3977.00   3977.00
mean    143.19       5.03    994.54
std     132.26       2.26    535.64
min       1.00       1.00      4.16
25%      43.00       3.00    600.79
50%     104.00       5.00    921.99
75%     203.00       6.00   1333.69
max     725.00      15.00   3770.73


In [ ]:
# let's look at the distribution characteristics
print("Top 10 customers by Monetary value:")
print(rfm.nlargest(10, 'Monetary')[['CustomerID','Recency','Frequency','Monetary']].to_string(index=False))
print()
print("Top 10 most frequent buyers:")
print(rfm.nlargest(10, 'Frequency')[['CustomerID','Recency','Frequency','Monetary']].to_string(index=False))

Top 10 customers by Monetary value:
CustomerID  Recency  Frequency  Monetary
    C13178        4         15   3770.73
    C10751       15         15   3630.73
    C13379      127         14   3543.74
    C10189       30         11   3485.57
    C11382       15         14   3410.96
    C13494        9         11   3364.38
    C11003        5         14   2969.36
    C12864       11         12   2943.16
    C10749       65         10   2880.94
    C12578      107         11   2838.53

Top 10 most frequent buyers:
CustomerID  Recency  Frequency  Monetary
    C10751       15         15   3630.73
    C13178        4         15   3770.73
    C11003        5         14   2969.36
    C11382       15         14   3410.96
    C13379      127         14   3543.74
    C10816        2         13   2398.49
    C12156       30         13   2551.30
    C12403       18         13   2464.23
    C12973       52         13   2543.39
    C13191       33         13   2189.11


## MODULE 4 — RFM Scoring & Rule-Based Segmentation

In [ ]:
# score each RFM dimension on a 1–5 scale using quintiles
# for Recency, lower is better (bought recently = score 5)
rfm['R_Score'] = pd.qcut(rfm['Recency'],   q=5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'].rank(method='first'),  q=5, labels=[1,2,3,4,5]).astype(int)

# combined RFM score string and total score
rfm['RFM_Score']  = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)
rfm['Total_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

print("Score distribution (R / F / M):")
for col in ['R_Score','F_Score','M_Score']:
    print(f"  {col}: {rfm[col].value_counts().sort_index().to_dict()}")
print()
print(rfm[['CustomerID','Recency','Frequency','Monetary','R_Score','F_Score','M_Score','Total_Score']].head(10).to_string(index=False))

Score distribution (R / F / M):
  R_Score: {1: 795, 2: 782, 3: 807, 4: 780, 5: 813}
  F_Score: {1: 796, 2: 795, 3: 795, 4: 795, 5: 796}
  M_Score: {1: 796, 2: 795, 3: 795, 4: 795, 5: 796}

CustomerID  Recency  Frequency  Monetary  R_Score  F_Score  M_Score  Total_Score
    C10000      309          8   1527.58        1        5        5           11
    C10001       40          4    822.89        4        2        3            9
    C10002       23          4    481.82        5        2        1            8
    C10003       49          5    876.23        4        3        3           10
    C10004       31         10   2238.64        5        5        5           15
    C10005        5          3    891.91        5        1        3            9
    C10006      303          3    719.94        1        1        2            4
    C10007       73          7   1513.43        4        4        5           13
    C10008       85          4    630.94        3        2        2            7
 

In [ ]:
# rule-based customer segmentation using RFM scores
def rfm_segment(row):
    r, f, m = row['R_Score'], row['F_Score'], row['M_Score']
    if r >= 4 and f >= 4 and m >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New Customers'
    elif r >= 3 and f <= 2 and m >= 3:
        return 'Potential Loyalists'
    elif r == 3 and f == 3:
        return 'Need Attention'
    elif r <= 2 and f >= 3 and m >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2 and m <= 2:
        return 'Lost Customers'
    else:
        return 'Hibernating'

rfm['Segment'] = rfm.apply(rfm_segment, axis=1)

print("Customer count per segment:")
seg_counts = rfm['Segment'].value_counts().reset_index()
seg_counts.columns = ['Segment','Count']
seg_counts['Percentage'] = (seg_counts['Count'] / len(rfm) * 100).round(1)
print(seg_counts.to_string(index=False))

Customer count per segment:
            Segment  Count  Percentage
    Loyal Customers    986        24.8
     Lost Customers    717        18.0
          Champions    702        17.7
            At Risk    555        14.0
        Hibernating    536        13.5
      New Customers    407        10.2
Potential Loyalists     74         1.9


In [ ]:
# segment-level summary — revenue, recency, frequency per group
seg_summary = rfm.groupby('Segment').agg(
    Customers      = ('CustomerID','count'),
    Avg_Recency    = ('Recency',   'mean'),
    Avg_Frequency  = ('Frequency', 'mean'),
    Total_Revenue  = ('Monetary',  'sum'),
    Avg_Revenue    = ('Monetary',  'mean')
).round(2).reset_index().sort_values('Total_Revenue', ascending=False)

seg_summary['Revenue_Share_pct'] = (seg_summary['Total_Revenue'] / seg_summary['Total_Revenue'].sum() * 100).round(1)
print("Segment Business Summary:")
print(seg_summary.to_string(index=False))

Segment Business Summary:
            Segment  Customers  Avg_Recency  Avg_Frequency  Total_Revenue  Avg_Revenue  Revenue_Share_pct
          Champions        702        33.46           7.72     1137459.73      1620.31               28.8
    Loyal Customers        986        71.07           5.90     1095420.31      1110.97               27.7
            At Risk        555       225.53           6.17      720716.43      1298.59               18.2
        Hibernating        536       187.88           3.64      363564.94       678.29                9.2
     Lost Customers        717       317.45           2.54      311650.84       434.66                7.9
      New Customers        407        36.55           3.20      254854.73       626.18                6.4
Potential Loyalists         74       101.78           3.58       71603.38       967.61                1.8


## MODULE 5 — K-Means Clustering

In [ ]:
# prepare features for clustering — log transform to reduce skew
rfm_cluster = rfm[['Recency','Frequency','Monetary']].copy()
rfm_cluster['Log_Recency']   = np.log1p(rfm_cluster['Recency'])
rfm_cluster['Log_Frequency'] = np.log1p(rfm_cluster['Frequency'])
rfm_cluster['Log_Monetary']  = np.log1p(rfm_cluster['Monetary'])

features = ['Log_Recency','Log_Frequency','Log_Monetary']

# standardize so no single feature dominates due to scale
scaler     = StandardScaler()
X_scaled   = scaler.fit_transform(rfm_cluster[features])

print("Feature matrix ready:")
print(f"  Shape   : {X_scaled.shape}")
print(f"  Means   : {X_scaled.mean(axis=0).round(4)}")
print(f"  Std devs: {X_scaled.std(axis=0).round(4)}")

Feature matrix ready:
  Shape   : (3977, 3)
  Means   : [0. 0. 0.]
  Std devs: [1. 1. 1.]


In [ ]:
# Elbow method — find the sweet spot for number of clusters
inertia_vals   = []
silhouette_vals = []
k_range = range(2, 11)

for k in k_range:
    km  = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_vals.append(km.inertia_)
    sil = silhouette_score(X_scaled, km.labels_, sample_size=min(2000, len(X_scaled)))
    silhouette_vals.append(round(sil, 4))

print("Elbow & Silhouette Analysis:")
print(f"{'K':>3}  {'Inertia':>12}  {'Silhouette':>12}")
print("─" * 32)
for k, iner, sil in zip(k_range, inertia_vals, silhouette_vals):
    print(f"{k:>3}  {iner:>12,.1f}  {sil:>12.4f}")

Elbow & Silhouette Analysis:
  K       Inertia    Silhouette
────────────────────────────────
  2       6,808.4        0.3771
  3       5,116.9        0.3115
  4       4,020.0        0.3179
  5       3,442.2        0.3000
  6       3,006.2        0.2754
  7       2,720.9        0.2837
  8       2,459.2        0.2850
  9       2,252.6        0.2693
 10       2,083.4        0.2775


In [ ]:
# based on elbow + silhouette, K=4 gives a good trade-off
OPTIMAL_K = 4

km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10, max_iter=300)
rfm['Cluster'] = km_final.fit_predict(X_scaled)

print(f"Final model: K={OPTIMAL_K} clusters")
print(f"Inertia    : {km_final.inertia_:,.2f}")
print(f"Silhouette : {silhouette_score(X_scaled, rfm['Cluster']):.4f}")
print()
print("Customers per cluster:")
print(rfm['Cluster'].value_counts().sort_index().to_string())

Final model: K=4 clusters
Inertia    : 4,020.05
Silhouette : 0.3130

Customers per cluster:
Cluster
0    1320
1     465
2    1391
3     801


In [ ]:
# profile each cluster using average RFM values
cluster_profile = rfm.groupby('Cluster').agg(
    Count         = ('CustomerID', 'count'),
    Avg_Recency   = ('Recency',    'mean'),
    Avg_Frequency = ('Frequency',  'mean'),
    Avg_Monetary  = ('Monetary',   'mean'),
    Total_Revenue = ('Monetary',   'sum')
).round(2).reset_index()

# assign descriptive labels based on R/F/M profile
def label_cluster(row):
    r, f, m = row['Avg_Recency'], row['Avg_Frequency'], row['Avg_Monetary']
    if r < 100 and f > 8 and m > 1500:
        return 'Champions'
    elif r < 200 and f > 5:
        return 'Loyal Customers'
    elif r > 300:
        return 'Lost / Churned'
    else:
        return 'Potential Loyalists'

cluster_profile['Cluster_Label'] = cluster_profile.apply(label_cluster, axis=1)
rfm = rfm.merge(cluster_profile[['Cluster','Cluster_Label']], on='Cluster', how='left')

print("Cluster Profiles:")
print(cluster_profile.to_string(index=False))

Cluster Profiles:
 Cluster  Count  Avg_Recency  Avg_Frequency  Avg_Monetary  Total_Revenue       Cluster_Label
       0   1320       121.20           6.94       1421.64     1876559.42     Loyal Customers
       1    465       280.14           1.93        262.50      122060.46 Potential Loyalists
       2   1391       190.58           3.71        713.93      993073.03 Potential Loyalists
       3    801        17.60           5.98       1202.97      963577.45     Loyal Customers


In [ ]:
# PCA for 2D visualization of clusters
pca      = PCA(n_components=2, random_state=42)
pca_data = pca.fit_transform(X_scaled)
rfm['PCA_1'] = pca_data[:, 0]
rfm['PCA_2'] = pca_data[:, 1]

print(f"PCA variance explained: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
print(f"Total variance captured: {pca.explained_variance_ratio_.sum():.1%}")

PCA variance explained: PC1=67.9%, PC2=26.1%
Total variance captured: 94.0%


## MODULE 6 — Business Insights & Cluster Interpretation

In [ ]:
# detailed cluster interpretation and marketing recommendations
print("=" * 70)
print("  CUSTOMER CLUSTER INSIGHTS & MARKETING RECOMMENDATIONS")
print("=" * 70)

for _, row in cluster_profile.iterrows():
    print(f"\nCluster {int(row['Cluster'])} — {row['Cluster_Label']}")
    print(f"  Customers      : {int(row['Count']):,}")
    print(f"  Avg Recency    : {row['Avg_Recency']:.0f} days since last purchase")
    print(f"  Avg Frequency  : {row['Avg_Frequency']:.1f} transactions")
    print(f"  Avg Spend      : ${row['Avg_Monetary']:,.2f}")
    print(f"  Total Revenue  : ${row['Total_Revenue']:,.2f}")

print("\n" + "─" * 70)
print("Revenue contribution by cluster:")
for _, row in cluster_profile.iterrows():
    share = row['Total_Revenue'] / cluster_profile['Total_Revenue'].sum() * 100
    print(f"  Cluster {int(row['Cluster'])} ({row['Cluster_Label']:20s}): {share:.1f}%")

  CUSTOMER CLUSTER INSIGHTS & MARKETING RECOMMENDATIONS

Cluster 0 — Loyal Customers
  Customers      : 1,320
  Avg Recency    : 121 days since last purchase
  Avg Frequency  : 6.9 transactions
  Avg Spend      : $1,421.64
  Total Revenue  : $1,876,559.42

Cluster 1 — Potential Loyalists
  Customers      : 465
  Avg Recency    : 280 days since last purchase
  Avg Frequency  : 1.9 transactions
  Avg Spend      : $262.50
  Total Revenue  : $122,060.46

Cluster 2 — Potential Loyalists
  Customers      : 1,391
  Avg Recency    : 191 days since last purchase
  Avg Frequency  : 3.7 transactions
  Avg Spend      : $713.93
  Total Revenue  : $993,073.03

Cluster 3 — Loyal Customers
  Customers      : 801
  Avg Recency    : 18 days since last purchase
  Avg Frequency  : 6.0 transactions
  Avg Spend      : $1,202.97
  Total Revenue  : $963,577.45

──────────────────────────────────────────────────────────────────────
Revenue contribution by cluster:
  Cluster 0 (Loyal Customers     ): 47.4%
  Cl

In [ ]:
# overall business KPIs
total_customers = len(rfm)
total_revenue   = rfm['Monetary'].sum()
avg_revenue     = rfm['Monetary'].mean()
avg_frequency   = rfm['Frequency'].mean()
avg_recency     = rfm['Recency'].mean()
top_10_revenue  = rfm.nlargest(int(total_customers * 0.10), 'Monetary')['Monetary'].sum()

print("=" * 55)
print("  OVERALL BUSINESS KPI SUMMARY")
print("=" * 55)
print(f"  Total Customers          : {total_customers:,}")
print(f"  Total Revenue            : ${total_revenue:,.2f}")
print(f"  Average Revenue/Customer : ${avg_revenue:,.2f}")
print(f"  Average Purchase Frequency: {avg_frequency:.1f} orders")
print(f"  Average Recency          : {avg_recency:.0f} days")
print(f"  Top 10% Customer Revenue : ${top_10_revenue:,.2f} ({top_10_revenue/total_revenue:.1%} of total)")
print("=" * 55)

  OVERALL BUSINESS KPI SUMMARY
  Total Customers          : 3,977
  Total Revenue            : $3,955,270.36
  Average Revenue/Customer : $994.54
  Average Purchase Frequency: 5.0 orders
  Average Recency          : 143 days
  Top 10% Customer Revenue : $816,809.17 (20.7% of total)


## MODULE 7 — Python Visualization Dashboard

In [ ]:
# Chart 1 — RFM Score Distribution (3-panel histogram)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('RFM Metric Distributions', fontsize=16, fontweight='bold', y=1.02)

rfm_metrics = [
    ('Recency',   'Days Since Last Purchase', PALETTE[0]),
    ('Frequency', 'Number of Transactions',   PALETTE[1]),
    ('Monetary',  'Total Spend ($)',           PALETTE[2])
]
for ax, (col, label, color) in zip(axes, rfm_metrics):
    data = rfm[col]
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5, label=f'Median: {data.median():.0f}')
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel(label, fontsize=10)
    ax.set_ylabel('Number of Customers', fontsize=10)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('chart1_rfm_distribution.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart1_rfm_distribution.png")

Saved: chart1_rfm_distribution.png


In [ ]:
# Chart 2 — Segment Customer Count (horizontal bar)
seg_data = rfm['Segment'].value_counts().reset_index()
seg_data.columns = ['Segment','Count']
seg_data = seg_data.sort_values('Count')

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = [PALETTE[i % len(PALETTE)] for i in range(len(seg_data))]
bars = ax.barh(seg_data['Segment'], seg_data['Count'], color=colors_bar, edgecolor='white', height=0.6)
for bar, val in zip(bars, seg_data['Count']):
    ax.text(bar.get_width() + 10, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10, fontweight='bold')
ax.set_title('Customer Count by RFM Segment', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Customers', fontsize=11)
ax.set_xlim(0, seg_data['Count'].max() * 1.15)
plt.tight_layout()
plt.savefig('chart2_segment_count.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart2_segment_count.png")

Saved: chart2_segment_count.png


In [ ]:
# Chart 3 — Revenue Contribution by Segment (donut chart)
seg_rev = rfm.groupby('Segment')['Monetary'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 7))
wedges, texts, autotexts = ax.pie(
    seg_rev, labels=seg_rev.index, autopct='%1.1f%%',
    startangle=90, colors=PALETTE[:len(seg_rev)],
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2),
    pctdistance=0.78, labeldistance=1.1
)
for t in autotexts:
    t.set_fontsize(9)
    t.set_fontweight('bold')
ax.set_title('Revenue Share by Customer Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart3_segment_revenue_donut.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart3_segment_revenue_donut.png")

Saved: chart3_segment_revenue_donut.png


In [ ]:
# Chart 4 — K-Means Clusters in PCA Space (scatter)
fig, ax = plt.subplots(figsize=(10, 7))
cluster_labels = rfm[['Cluster','Cluster_Label']].drop_duplicates().set_index('Cluster')['Cluster_Label'].to_dict()

for k in sorted(rfm['Cluster'].unique()):
    subset = rfm[rfm['Cluster'] == k]
    ax.scatter(subset['PCA_1'], subset['PCA_2'],
               c=PALETTE[k], label=f"Cluster {k}: {cluster_labels[k]}",
               alpha=0.55, s=30, edgecolors='white', linewidths=0.3)

# plot centroids projected onto PCA space
pca_centroids = pca.transform(km_final.cluster_centers_)
ax.scatter(pca_centroids[:, 0], pca_centroids[:, 1],
           c='black', marker='X', s=200, zorder=5, label='Centroids')
ax.set_title('K-Means Customer Clusters (PCA 2D projection)', fontsize=14, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
ax.legend(fontsize=9, loc='upper right')
plt.tight_layout()
plt.savefig('chart4_kmeans_pca.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart4_kmeans_pca.png")

Saved: chart4_kmeans_pca.png


In [ ]:
# Chart 5 — Elbow + Silhouette combined
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(list(k_range), inertia_vals, 'o-', color=PALETTE[0], linewidth=2, markersize=7)
ax1.axvline(OPTIMAL_K, color='red', linestyle='--', linewidth=1.5, label=f'Chosen K={OPTIMAL_K}')
ax1.set_title('Elbow Method — WCSS vs K', fontsize=13, fontweight='bold')
ax1.set_xlabel('Number of Clusters (K)', fontsize=11)
ax1.set_ylabel('Within-Cluster Sum of Squares', fontsize=11)
ax1.legend(); ax1.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

ax2.plot(list(k_range), silhouette_vals, 's-', color=PALETTE[2], linewidth=2, markersize=7)
ax2.axvline(OPTIMAL_K, color='red', linestyle='--', linewidth=1.5, label=f'Chosen K={OPTIMAL_K}')
ax2.set_title('Silhouette Score vs K', fontsize=13, fontweight='bold')
ax2.set_xlabel('Number of Clusters (K)', fontsize=11)
ax2.set_ylabel('Silhouette Score', fontsize=11)
ax2.legend(); ax2.xaxis.set_major_locator(plt.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('chart5_elbow_silhouette.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart5_elbow_silhouette.png")

Saved: chart5_elbow_silhouette.png


In [ ]:
# Chart 6 — RFM Score Heatmap (R vs F, colored by M)
rfm_pivot = rfm.groupby(['R_Score','F_Score'])['Monetary'].mean().reset_index()
rfm_heat  = rfm_pivot.pivot(index='R_Score', columns='F_Score', values='Monetary')

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(rfm_heat, annot=True, fmt='.0f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label':'Avg Monetary ($)'},
            ax=ax, annot_kws={'size':10})
ax.set_title('Average Monetary Value — Recency Score vs Frequency Score', fontsize=13, fontweight='bold')
ax.set_xlabel('Frequency Score (1=Low → 5=High)', fontsize=11)
ax.set_ylabel('Recency Score (1=Low → 5=High)', fontsize=11)
plt.tight_layout()
plt.savefig('chart6_rfm_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart6_rfm_heatmap.png")

Saved: chart6_rfm_heatmap.png


In [ ]:
# Chart 7 — Cluster Radar / Spider Chart (avg RFM per cluster)
from matplotlib.patches import FancyArrowPatch

cluster_radar = rfm.groupby('Cluster')[['R_Score','F_Score','M_Score']].mean().reset_index()
labels_radar  = ['Recency Score', 'Frequency Score', 'Monetary Score']
num_vars      = len(labels_radar)
angles        = np.linspace(0, 2*np.pi, num_vars, endpoint=False).tolist()
angles       += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for _, row in cluster_radar.iterrows():
    k     = int(row['Cluster'])
    vals  = [row['R_Score'], row['F_Score'], row['M_Score']]
    vals += vals[:1]
    ax.plot(angles, vals, 'o-', linewidth=2, color=PALETTE[k],
            label=f"Cluster {k}: {cluster_labels[k]}")
    ax.fill(angles, vals, alpha=0.12, color=PALETTE[k])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels_radar, fontsize=12)
ax.set_ylim(0, 5.5)
ax.set_yticks([1, 2, 3, 4, 5]); ax.set_yticklabels(['1','2','3','4','5'], fontsize=8)
ax.set_title('Cluster RFM Profile — Radar Chart', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=10)
plt.tight_layout()
plt.savefig('chart7_radar.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart7_radar.png")

Saved: chart7_radar.png


In [ ]:
# Chart 8 — Top 10 Customers by Revenue (bar chart)
top10_cust = rfm.nlargest(10, 'Monetary')[['CustomerID','Monetary','Segment','Cluster_Label']].copy()

fig, ax = plt.subplots(figsize=(11, 6))
colors_cust = [PALETTE[i % len(PALETTE)] for i in range(len(top10_cust))]
bars = ax.bar(range(len(top10_cust)), top10_cust['Monetary'], color=colors_cust, edgecolor='white', width=0.65)
for i, (bar, row) in enumerate(zip(bars, top10_cust.itertuples())):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'${row.Monetary:,.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()/2,
            row.Cluster_Label, ha='center', va='center', fontsize=7, color='white', fontweight='bold', rotation=90)
ax.set_xticks(range(len(top10_cust)))
ax.set_xticklabels(top10_cust['CustomerID'], rotation=30, ha='right', fontsize=9)
ax.set_title('Top 10 Customers by Total Revenue', fontsize=14, fontweight='bold')
ax.set_ylabel('Total Revenue ($)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:,.0f}'))
plt.tight_layout()
plt.savefig('chart8_top10_customers.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart8_top10_customers.png")

Saved: chart8_top10_customers.png


In [ ]:
# Chart 9 — Monthly Revenue Trend
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
monthly_rev     = df.groupby('YearMonth')['TotalPrice'].sum().reset_index()
monthly_rev['YearMonth_str'] = monthly_rev['YearMonth'].astype(str)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(len(monthly_rev)), monthly_rev['TotalPrice'],
                alpha=0.25, color=PALETTE[1])
ax.plot(range(len(monthly_rev)), monthly_rev['TotalPrice'],
        'o-', color=PALETTE[1], linewidth=2, markersize=5)
ax.set_xticks(range(len(monthly_rev)))
ax.set_xticklabels(monthly_rev['YearMonth_str'], rotation=45, ha='right', fontsize=8)
ax.set_title('Monthly Revenue Trend', fontsize=14, fontweight='bold')
ax.set_ylabel('Revenue ($)', fontsize=11)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
plt.tight_layout()
plt.savefig('chart9_monthly_trend.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart9_monthly_trend.png")

Saved: chart9_monthly_trend.png


In [ ]:
# Chart 10 — Cluster Revenue Contribution (grouped bar)
clust_rev = rfm.groupby('Cluster').agg(
    Revenue  = ('Monetary','sum'),
    Customers= ('CustomerID','count')
).reset_index()
clust_rev['Label'] = clust_rev['Cluster'].map(cluster_labels)

fig, ax1 = plt.subplots(figsize=(10, 6))
x = np.arange(len(clust_rev))
bars1 = ax1.bar(x - 0.2, clust_rev['Revenue'], width=0.38,
                color=[PALETTE[k] for k in clust_rev['Cluster']], label='Revenue ($)', edgecolor='white')
ax1.set_ylabel('Total Revenue ($)', fontsize=11, color=PALETTE[0])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v/1000:.0f}K'))

ax2 = ax1.twinx()
ax2.bar(x + 0.2, clust_rev['Customers'], width=0.38,
        color=[PALETTE[k] for k in clust_rev['Cluster']], alpha=0.45, label='Customers', edgecolor='white')
ax2.set_ylabel('Number of Customers', fontsize=11, color=PALETTE[2])

ax1.set_xticks(x)
ax1.set_xticklabels([f"C{k}\n{l}" for k, l in zip(clust_rev['Cluster'], clust_rev['Label'])], fontsize=10)
ax1.set_title('Revenue & Customer Count by K-Means Cluster', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('chart10_cluster_revenue.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved: chart10_cluster_revenue.png")

Saved: chart10_cluster_revenue.png


## MODULE 8 — Excel Report Generator (Multi-Sheet Dashboard)

In [ ]:
!pip install XlsxWriter --quiet
print("XlsxWriter ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.4 MB/s eta 0:00:00
XlsxWriter ready


In [ ]:
# ── STEP 1: prepare all data tables for the report ──
kpi_table = pd.DataFrame({
    'KPI Metric': [
        'Total Customers Analyzed',
        'Total Revenue',
        'Average Revenue per Customer',
        'Average Purchase Frequency',
        'Average Recency (days)',
        'Top 10% Customer Revenue Share',
        'Number of Segments (Rule-Based)',
        'Number of Clusters (K-Means)',
        'K-Means Silhouette Score',
        'Champion Customers'
    ],
    'Value': [
        f"{total_customers:,}",
        f"${total_revenue:,.2f}",
        f"${avg_revenue:,.2f}",
        f"{avg_frequency:.1f} orders",
        f"{avg_recency:.0f} days",
        f"{top_10_revenue/total_revenue:.1%}",
        f"{rfm['Segment'].nunique()}",
        f"{OPTIMAL_K}",
        f"{silhouette_score(X_scaled, rfm['Cluster']):.4f}",
        f"{(rfm['Segment']=='Champions').sum():,}"
    ]
})

rfm_export = rfm[['CustomerID','Recency','Frequency','Monetary',
                   'R_Score','F_Score','M_Score','Total_Score',
                   'Segment','Cluster','Cluster_Label']].copy()

print("Data tables ready for export.")
print(f"  KPI table     : {len(kpi_table)} rows")
print(f"  RFM full table: {len(rfm_export):,} rows")
print(f"  Segment summary: {len(seg_summary)} rows")
print(f"  Cluster profile: {len(cluster_profile)} rows")

Data tables ready for export.
  KPI table     : 10 rows
  RFM full table: 3,977 rows
  Segment summary: 7 rows
  Cluster profile: 4 rows


In [ ]:
# ── STEP 2: build the Excel workbook ──
import xlsxwriter, os

report_file = 'Customer_Segmentation_RFM_Report.xlsx'

with pd.ExcelWriter(report_file, engine='xlsxwriter',
                    engine_kwargs={'options': {'nan_inf_to_errors': True}}) as writer:
    wb = writer.book

    # ── FORMAT DEFINITIONS ──────────────────────────────────────────────
    fmt_title  = wb.add_format({'bold': True, 'font_size': 16, 'font_color': '#2C3E50', 'align': 'left'})
    fmt_sub    = wb.add_format({'bold': True, 'font_size': 11, 'font_color': '#7F8C8D', 'align': 'left'})
    fmt_hdr    = wb.add_format({'bold': True, 'bg_color': '#2C3E50', 'font_color': '#FFFFFF',
                                'border': 1, 'align': 'center', 'valign': 'vcenter', 'text_wrap': True})
    fmt_kpi_lbl= wb.add_format({'bold': True, 'bg_color': '#EBF5FB', 'border': 1, 'font_size': 11})
    fmt_kpi_val= wb.add_format({'bold': True, 'bg_color': '#FDFEFE', 'border': 1, 'font_size': 11,
                                'font_color': '#2980B9', 'align': 'center'})
    fmt_even   = wb.add_format({'bg_color': '#F2F3F4', 'border': 1, 'align': 'center'})
    fmt_odd    = wb.add_format({'bg_color': '#FDFEFE', 'border': 1, 'align': 'center'})
    fmt_money  = wb.add_format({'num_format': '$#,##0.00', 'border': 1, 'align': 'center'})
    fmt_int    = wb.add_format({'num_format': '#,##0',     'border': 1, 'align': 'center'})
    fmt_pct    = wb.add_format({'num_format': '0.0%',      'border': 1, 'align': 'center'})
    fmt_seg    = {
        'Champions'         : wb.add_format({'bg_color':'#1ABC9C','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
        'Loyal Customers'   : wb.add_format({'bg_color':'#2980B9','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
        'New Customers'     : wb.add_format({'bg_color':'#27AE60','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
        'Potential Loyalists': wb.add_format({'bg_color':'#F39C12','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
        'Need Attention'    : wb.add_format({'bg_color':'#E67E22','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
        'At Risk'           : wb.add_format({'bg_color':'#E74C3C','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
        'Hibernating'       : wb.add_format({'bg_color':'#95A5A6','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
        'Lost Customers'    : wb.add_format({'bg_color':'#7F8C8D','font_color':'#FFFFFF','bold':True,'border':1,'align':'center'}),
    }

    #   SHEET 1 — EXECUTIVE SUMMARY

    ws1 = wb.add_worksheet('Executive Summary')
    writer.sheets['Executive Summary'] = ws1
    ws1.hide_gridlines(2)
    ws1.set_column('A:A', 38)
    ws1.set_column('B:B', 28)
    ws1.set_column('C:H', 18)

    ws1.write('A1', 'Customer Segmentation RFM Analysis', fmt_title)
    ws1.write('A2', 'Online Retail — Executive Summary Report', fmt_sub)
    ws1.write('A3', 'Dataset: Online Retail II (Kaggle / UCI ML Repository)  |  Method: RFM Scoring + K-Means Clustering',
              wb.add_format({'font_size': 9, 'font_color': '#95A5A6', 'italic': True}))

    ws1.write('A5', 'KEY PERFORMANCE INDICATORS', wb.add_format({'bold': True, 'font_size': 13, 'font_color': '#2C3E50'}))
    ws1.write('A6', 'Metric', fmt_hdr)
    ws1.write('B6', 'Value',  fmt_hdr)
    for i, (_, row) in enumerate(kpi_table.iterrows()):
        r = 6 + i
        ws1.write(r, 0, row['KPI Metric'], fmt_kpi_lbl)
        ws1.write(r, 1, row['Value'],       fmt_kpi_val)

    # embed charts into executive summary
    row_offset = 6 + len(kpi_table) + 2
    ws1.write(row_offset, 0, 'VISUAL INSIGHTS', wb.add_format({'bold': True, 'font_size': 13, 'font_color': '#2C3E50'}))

    for col_off, chart_file in enumerate(['chart2_segment_count.png', 'chart3_segment_revenue_donut.png',
                                          'chart4_kmeans_pca.png', 'chart10_cluster_revenue.png']):
        if os.path.exists(chart_file):
            c_row = row_offset + 1 + (col_off // 2) * 18
            c_col = (col_off % 2) * 6
            ws1.insert_image(c_row, c_col, chart_file, {'x_scale': 0.72, 'y_scale': 0.72})

    #   SHEET 2 — RFM DISTRIBUTIONS

    ws2 = wb.add_worksheet('RFM Distributions')
    writer.sheets['RFM Distributions'] = ws2
    ws2.hide_gridlines(2)
    ws2.set_column('A:H', 16)

    ws2.write('A1', 'RFM Distribution Analysis', fmt_title)
    ws2.write('A2', 'Recency · Frequency · Monetary Score Distributions', fmt_sub)

    for col_off, chart_file in enumerate(['chart1_rfm_distribution.png', 'chart6_rfm_heatmap.png',
                                          'chart5_elbow_silhouette.png', 'chart7_radar.png']):
        if os.path.exists(chart_file):
            r_pos = 3 + (col_off // 2) * 20
            c_pos = (col_off % 2) * 9
            ws2.insert_image(r_pos, c_pos, chart_file, {'x_scale': 0.80, 'y_scale': 0.80})


    #  SHEET 3 — SEGMENT ANALYSIS

    ws3 = wb.add_worksheet('Segment Analysis')
    writer.sheets['Segment Analysis'] = ws3
    ws3.hide_gridlines(2)
    ws3.set_column('A:A', 22)
    ws3.set_column('B:G', 18)

    ws3.write('A1', 'RFM Customer Segment Analysis', fmt_title)
    ws3.write('A2', 'Rule-Based Segmentation Summary', fmt_sub)

    cols3 = ['Segment','Customers','Avg Recency (days)','Avg Frequency','Total Revenue ($)','Avg Revenue ($)','Revenue Share (%)']
    for c, h in enumerate(cols3):
        ws3.write(3, c, h, fmt_hdr)
        ws3.set_column(c, c, max(18, len(h)+4))

    for r, (_, row) in enumerate(seg_summary.iterrows()):
        row_idx = 4 + r
        sfmt = fmt_seg.get(row['Segment'], fmt_even)
        ws3.write(row_idx, 0, row['Segment'],         sfmt)
        ws3.write(row_idx, 1, row['Customers'],        fmt_int)
        ws3.write(row_idx, 2, round(row['Avg_Recency'],1),   fmt_even)
        ws3.write(row_idx, 3, round(row['Avg_Frequency'],1), fmt_even)
        ws3.write(row_idx, 4, row['Total_Revenue'],    fmt_money)
        ws3.write(row_idx, 5, row['Avg_Revenue'],      fmt_money)
        ws3.write(row_idx, 6, row['Revenue_Share_pct']/100, fmt_pct)

    img_row = 4 + len(seg_summary) + 2
    ws3.write(img_row, 0, 'Segment Visualizations', wb.add_format({'bold': True, 'font_size': 13, 'font_color': '#2C3E50'}))
    for i, ch in enumerate(['chart2_segment_count.png', 'chart3_segment_revenue_donut.png']):
        if os.path.exists(ch):
            ws3.insert_image(img_row + 1, i * 9, ch, {'x_scale': 0.78, 'y_scale': 0.78})


    #  SHEET 4 — CLUSTER ANALYSIS

    ws4 = wb.add_worksheet('K-Means Clusters')
    writer.sheets['K-Means Clusters'] = ws4
    ws4.hide_gridlines(2)
    ws4.set_column('A:A', 14)
    ws4.set_column('B:F', 22)

    ws4.write('A1', 'K-Means Clustering Results', fmt_title)
    ws4.write('A2', f'Optimal K = {OPTIMAL_K} clusters identified via Elbow + Silhouette analysis', fmt_sub)

    cluster_cols = ['Cluster','Cluster_Label','Count','Avg Recency','Avg Frequency','Total Revenue ($)','Avg Revenue ($)']
    for c, h in enumerate(cluster_cols):
        ws4.write(3, c, h, fmt_hdr)

    for r, (_, row) in enumerate(cluster_profile.iterrows()):
        ri = 4 + r
        fmt_c = wb.add_format({'bg_color': PALETTE[int(row['Cluster'])], 'font_color':'#FFFFFF',
                                'bold': True, 'border': 1, 'align': 'center'})
        ws4.write(ri, 0, int(row['Cluster']),       fmt_c)
        ws4.write(ri, 1, row['Cluster_Label'],       fmt_even)
        ws4.write(ri, 2, int(row['Count']),          fmt_int)
        ws4.write(ri, 3, round(row['Avg_Recency'],1), fmt_even)
        ws4.write(ri, 4, round(row['Avg_Frequency'],1),fmt_even)
        ws4.write(ri, 5, row['Total_Revenue'],        fmt_money)
        ws4.write(ri, 6, round(row['Avg_Monetary'],2), fmt_money)

    img_r4 = 4 + len(cluster_profile) + 2
    ws4.write(img_r4, 0, 'Cluster Visualizations', wb.add_format({'bold': True, 'font_size': 13, 'font_color': '#2C3E50'}))
    for i, ch in enumerate(['chart4_kmeans_pca.png', 'chart7_radar.png', 'chart5_elbow_silhouette.png', 'chart10_cluster_revenue.png']):
        if os.path.exists(ch):
            r_pos = img_r4 + 1 + (i // 2) * 18
            c_pos = (i % 2) * 9
            ws4.insert_image(r_pos, c_pos, ch, {'x_scale': 0.78, 'y_scale': 0.78})


    #   SHEET 5 — TREND & TOP CUSTOMERS

    ws5 = wb.add_worksheet('Trends & Top Customers')
    writer.sheets['Trends & Top Customers'] = ws5
    ws5.hide_gridlines(2)
    ws5.set_column('A:F', 20)

    ws5.write('A1', 'Revenue Trends & Top Customers', fmt_title)
    ws5.write('A2', 'Monthly revenue trend and highest-value customers', fmt_sub)

    # monthly revenue table
    ws5.write('A4', 'Monthly Revenue', wb.add_format({'bold': True, 'font_size': 12, 'font_color': '#2C3E50'}))
    for c, h in enumerate(['Year-Month', 'Revenue ($)']):
        ws5.write(5, c, h, fmt_hdr)
    for r, row in monthly_rev.iterrows():
        ws5.write(6+r, 0, row['YearMonth_str'], fmt_even if r%2==0 else fmt_odd)
        ws5.write(6+r, 1, row['TotalPrice'],     fmt_money)

    # top 10 customers table
    top10_col_start = 3
    ws5.write(4, top10_col_start, 'Top 10 Customers by Revenue', wb.add_format({'bold': True, 'font_size': 12, 'font_color': '#2C3E50'}))
    for c, h in enumerate(['Customer ID','Revenue ($)','Segment','Cluster']):
        ws5.write(5, top10_col_start+c, h, fmt_hdr)
    for r, row in enumerate(top10_cust.itertuples()):
        ri = 6 + r
        fmt_r = fmt_even if r % 2 == 0 else fmt_odd
        ws5.write(ri, top10_col_start+0, row.CustomerID,    fmt_r)
        ws5.write(ri, top10_col_start+1, row.Monetary,      fmt_money)
        ws5.write(ri, top10_col_start+2, row.Segment,       fmt_seg.get(row.Segment, fmt_r))
        ws5.write(ri, top10_col_start+3, row.Cluster_Label, fmt_r)

    img_r5 = max(6 + len(monthly_rev), 6 + len(top10_cust)) + 2
    for i, ch in enumerate(['chart9_monthly_trend.png', 'chart8_top10_customers.png']):
        if os.path.exists(ch):
            ws5.insert_image(img_r5, i*9, ch, {'x_scale': 0.78, 'y_scale': 0.78})


    #   SHEET 6 — FULL RFM DATA

    rfm_export.to_excel(writer, sheet_name='Full RFM Data', index=False)
    ws6 = writer.sheets['Full RFM Data']
    ws6.hide_gridlines(2)
    ws6.set_column('A:K', 18)
    for c, h in enumerate(rfm_export.columns):
        ws6.write(0, c, h, fmt_hdr)

print(f"Excel report saved: {report_file}")
print(f"File size: {os.path.getsize(report_file)/1024:.1f} KB")

Excel report saved: Customer_Segmentation_RFM_Report.xlsx
File size: 1116.3 KB


## MODULE 9 — Final Summary & Business Recommendations

In [ ]:
# wrap-up summary
print("=" * 65)
print("  CUSTOMER SEGMENTATION PROJECT — COMPLETE SUMMARY")
print("=" * 65)

print("\n RFM ANALYSIS COMPLETE")
print(f"  Customers analyzed    : {len(rfm):,}")
print(f"  Rule-based segments   : {rfm['Segment'].nunique()}")
print(f"  K-Means clusters      : {OPTIMAL_K}")
print(f"  Silhouette score      : {silhouette_score(X_scaled, rfm['Cluster']):.4f}")

print("\n BUSINESS RECOMMENDATIONS BY SEGMENT")
recommendations = {
    'Champions'         : 'Reward them! Ask for reviews, enroll in loyalty programs.',
    'Loyal Customers'   : 'Upsell higher-value products. Appreciate them.',
    'Potential Loyalists': 'Offer membership / loyalty programs, recommend new products.',
    'New Customers'     : 'Provide onboarding support, share best sellers.',
    'Need Attention'    : 'Reactivate with limited-time offers. Share helpful resources.',
    'At Risk'           : 'Send personalized reactivation emails with discounts.',
    'Hibernating'       : 'Share relevant resources, offer seasonal discounts.',
    'Lost Customers'    : 'Ignore or run low-cost win-back campaigns.',
}
for seg, rec in recommendations.items():
    count = (rfm['Segment'] == seg).sum()
    print(f"  {seg:22s} ({count:4d} cust.) → {rec}")

print("\n OUTPUT FILES GENERATED")
charts = [f'chart{i}' for i in range(1, 11)]
for f in ['Customer_Segmentation_RFM_Report.xlsx', 'online_retail_rfm.csv'] +          [f'chart{i}_{n}.png' for i, n in enumerate(
              ['rfm_distribution','segment_count','segment_revenue_donut',
               'kmeans_pca','elbow_silhouette','rfm_heatmap','radar',
               'top10_customers','monthly_trend','cluster_revenue'], 1)]:
    import os
    exists = os.path.exists(f)
    print(f"  {'[OK]' if exists else '[--]'} {f}")

print("\n Project complete!")

  CUSTOMER SEGMENTATION PROJECT — COMPLETE SUMMARY

 RFM ANALYSIS COMPLETE
  Customers analyzed    : 3,977
  Rule-based segments   : 7
  K-Means clusters      : 4
  Silhouette score      : 0.3130

 BUSINESS RECOMMENDATIONS BY SEGMENT
  Champions              ( 702 cust.) → Reward them! Ask for reviews, enroll in loyalty programs.
  Loyal Customers        ( 986 cust.) → Upsell higher-value products. Appreciate them.
  Potential Loyalists    (  74 cust.) → Offer membership / loyalty programs, recommend new products.
  New Customers          ( 407 cust.) → Provide onboarding support, share best sellers.
  Need Attention         (   0 cust.) → Reactivate with limited-time offers. Share helpful resources.
  At Risk                ( 555 cust.) → Send personalized reactivation emails with discounts.
  Hibernating            ( 536 cust.) → Share relevant resources, offer seasonal discounts.
  Lost Customers         ( 717 cust.) → Ignore or run low-cost win-back campaigns.

 OUTPUT FILES GENERA